<a href="https://colab.research.google.com/github/kite121/Kaggle_competitions/blob/main/Refined_version_Kaggle_House.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
# Cell 0: Functions used for analysis

def hist_feature(df, feature_name, target=None):
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    feature = df[feature_name]

    sns.histplot(feature.dropna(), bins=40, kde=True, ax=axes[0])
    axes[0].set_title(f"{feature_name} (Raw)")

    # Безопасный log1p: только значения > -1
    log_values = feature[feature > -1].dropna()
    sns.histplot(np.log1p(log_values), bins=40, kde=True, ax=axes[1])
    axes[1].set_title(f"{feature_name} (Log1p)")

    plt.tight_layout()
    plt.show()


def scatter_feature(df, feature_name, target):
    sns.scatterplot(data=df, x=feature_name, y=target)
    plt.show()


def binned_feature(df, feature_name, target, q=10):
    tmp = df[[feature_name, target]].dropna().copy()
    tmp["bin"] = pd.qcut(tmp[feature_name], q=q, duplicates="drop")
    sns.boxplot(data=tmp, x="bin", y=target)
    plt.xticks(rotation=45)
    plt.show()


def correlation_feature(df, feature_name, target):
    pear = df[[feature_name, target]].corr(method="pearson").iloc[0, 1]
    spear = df[[feature_name, target]].corr(method="spearman").iloc[0, 1]
    print(f"Pearson: {pear:.4f}, Spearman: {spear:.4f}")


def box_plot_feature(df, feature_name):
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))

    df[[feature_name]].boxplot(ax=axes[0])
    axes[0].set_title("Raw")

    log_df = df[[feature_name]].copy()
    log_df = log_df[log_df[feature_name] > -1]
    log_df.apply(lambda x: np.log1p(x)).boxplot(ax=axes[1])
    axes[1].set_title("Log1p")

    plt.tight_layout()
    plt.show()


def ordinal_check(df, feature_name, target):
    s = df[feature_name].fillna("Missing")
    quality_pivot = (
        pd.pivot_table(
            pd.DataFrame({feature_name: s, target: df[target]}),
            index=feature_name,
            values=target,
            aggfunc=np.median,
        )
        .sort_values(target)
    )
    quality_pivot.plot(kind="bar")
    plt.show()
    return quality_pivot


def feature_counts(df, feature_name, normalize=False):
    return df[feature_name].value_counts(normalize=normalize, dropna=False)

In [53]:
# Cell 1: imports
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from pathlib import Path
from pprint import pprint

from sklearn.model_selection import KFold, cross_val_score, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler, RobustScaler, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.metrics import mean_squared_error, make_scorer
from sklearn.feature_selection import mutual_info_regression
from sklearn.neural_network import MLPRegressor
import numpy as np
from catboost import CatBoostRegressor, Pool
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

SEED = 42
np.random.seed(SEED)


In [3]:
# Cell 2: loading data
TRAIN_PATH = "train.csv"
TEST_PATH = "test.csv"

train_raw = pd.read_csv(TRAIN_PATH)
test_raw = pd.read_csv(TEST_PATH)


print(train_raw.shape, test_raw.shape)


(1460, 81) (1459, 80)


In [4]:
# Cell 3: Target diagnostics
y_raw = train_raw["SalePrice"]
y_log = np.log1p(y_raw)

print("SalePrice skew (raw):", round(y_raw.skew(), 3))
print("SalePrice skew (log1p):", round(y_log.skew(), 3))
print("Target quantiles:")
print(y_raw.quantile([0.5, 0.9, 0.95, 0.99]))

SalePrice skew (raw): 1.883
SalePrice skew (log1p): 0.121
Target quantiles:
0.50    163000.00
0.90    278000.00
0.95    326100.00
0.99    442567.01
Name: SalePrice, dtype: float64


In [5]:
# Cell 4: Feature engineering configs

NEIGHBORHOOD_TIERS = {
    "MeadowV": "VeryLow", "IDOTRR": "VeryLow", "BrDale": "VeryLow", "OldTown": "VeryLow", "Edwards": "VeryLow", "BrkSide": "VeryLow",
    "Sawyer": "Low", "Blueste": "Low", "SWISU": "Low", "NAmes": "Low", "NPkVill": "Low", "Mitchel": "Low",
    "SawyerW": "Medium", "Gilbert": "Medium", "NWAmes": "Medium", "Blmngtn": "Medium", "CollgCr": "Medium", "ClearCr": "Medium", "Crawfor": "Medium",
    "Veenker": "High", "Somerst": "High", "Timber": "High",
    "StoneBr": "VeryHigh", "NoRidge": "VeryHigh", "NridgHt": "VeryHigh",
}

DROP_AFTER_ENGINEERING = [
    "YearBuilt", "YearRemodAdd", "BsmtFinSF1", "BsmtFinSF2", "BsmtUnfSF",
    "1stFlrSF", "2ndFlrSF", "LowQualFinSF", "BsmtFullBath", "BsmtHalfBath",
    "FullBath", "HalfBath", "BedroomAbvGr", "KitchenAbvGr", "WoodDeckSF",
    "OpenPorchSF", "EnclosedPorch", "3SsnPorch", "ScreenPorch", "PoolArea",
    "MiscVal", "MoSold", "YrSold", "GarageYrBlt", "Alley", "Utilities",
]

LOG_ROBUST_NUM_CANDIDATES = [
    "LotFrontage", "LotArea", "MasVnrArea", "TotalBsmtSF", "GrLivArea", "GarageArea", "TotalPorchSF"
]

feature_report = pd.DataFrame(
    [
        ("EffectiveYear", "numeric", "max(YearBuilt, YearRemodAdd)"),
        ("HouseAge", "numeric", "YrSold - EffectiveYear"),
        ("GarageAge", "numeric", "YrSold - GarageYrBlt"),
        ("TotalBath_equiv", "numeric", "Full + BsmtFull + 0.5*(Half + BsmtHalf), cap<=4"),
        ("TotalPorchSF", "numeric", "WoodDeck + OpenPorch + 3SsnPorch + ScreenPorch"),
        ("BedroomShare", "numeric", "BedroomAbvGr / TotRmsAbvGrd"),
        ("OverallCondGrp", "ordinal_cat", "bin OverallCond -> Low/Medium/High"),
        ("NeighborhoodTier", "ordinal_cat", "manual tier mapping from Neighborhood"),
        ("HasAlley", "binary", "Alley is not missing"),
        ("HasFinishedBsmt", "binary", "BsmtFinSF1 + BsmtFinSF2 > 0"),
        ("HasBathBsmt", "binary", "Bsmt baths > 0"),
        ("HasBathGr", "binary", "ground-floor baths > 0"),
        ("HasKitchen", "binary", "KitchenAbvGr > 0"),
        ("HasPool", "binary", "PoolArea > 0"),
        ("HasMiscVal", "binary", "MiscVal > 0"),
        ("HasGarage", "binary", "GarageYrBlt is not missing"),
        ("HasMas", "binary", "MasVnrArea > 0"),
    ],
    columns=["Feature", "Type", "Rule"],
)

feature_report

,Feature,Type,Rule
0,EffectiveYear,numeric,"max(YearBuilt, YearRemodAdd)"
1,HouseAge,numeric,YrSold - EffectiveYear
2,GarageAge,numeric,YrSold - GarageYrBlt
3,TotalBath_equiv,numeric,"Full + BsmtFull + 0.5*(Half + BsmtHalf), cap<=4"
4,TotalPorchSF,numeric,WoodDeck + OpenPorch + 3SsnPorch + ScreenPorch
5,BedroomShare,numeric,BedroomAbvGr / TotRmsAbvGrd
6,OverallCondGrp,ordinal_cat,bin OverallCond -> Low/Medium/High
7,NeighborhoodTier,ordinal_cat,manual tier mapping from Neighborhood
8,HasAlley,binary,Alley is not missing
9,HasFinishedBsmt,binary,BsmtFinSF1 + BsmtFinSF2 > 0


In [ ]:
def hist_feature(feature_name):
  fig, axes = plt.subplots(1, 2, figsize = (10, 4))
  feature = df_train[feature_name]
  sns.histplot(feature, bins = 40, kde = True, ax = axes[0])
  axes[0].set_title("Raw")
  sns.histplot(np.log1p(feature), bins = 40, kde = True, ax = axes[1])
  axes[1].set_title("Log1p")
  plt.tight_layout()
  plt.show()

def scatter_feature(feature_name):
  sns.scatterplot(x= df_train[feature_name], y=df_train["SalePrice"])

def binned_feature(feature_name):
  df_train["bin"] = pd.qcut(df_train[feature_name], q=10, duplicates="drop")
  sns.boxplot(x="bin", y="SalePrice", data=df_train)
  plt.xticks(rotation=45)

def correlation_feature(feature_name):
  pear = df_train[[feature_name, "SalePrice"]].corr(method="pearson").iloc[0, 1]
  spear = df_train[[feature_name, "SalePrice"]].corr(method="spearman").iloc[0, 1]
  print(pear, spear)

In [8]:
# Cell 5: Main engineering function

def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    d = df.copy()

    # Temporal features
    d["EffectiveYear"] = d[["YearBuilt", "YearRemodAdd"]].max(axis=1)
    d["HouseAge"] = d["YrSold"] - d["EffectiveYear"]
    d["GarageAge"] = d["YrSold"] - d["GarageYrBlt"].fillna(d["YrSold"])

    # Binary indicators
    d["HasGarage"] = d["GarageYrBlt"].notna().astype(int)
    d["HasMas"] = (d["MasVnrArea"].fillna(0) > 0).astype(int)
    d["HasFinishedBsmt"] = ((d["BsmtFinSF1"].fillna(0) + d["BsmtFinSF2"].fillna(0)) > 0).astype(int)
    d["HasBathBsmt"] = ((d["BsmtFullBath"].fillna(0) + d["BsmtHalfBath"].fillna(0)) > 0).astype(int)
    d["HasBathGr"] = ((d["FullBath"].fillna(0) + d["HalfBath"].fillna(0)) > 0).astype(int)
    d["HasKitchen"] = (d["KitchenAbvGr"].fillna(0) > 0).astype(int)
    d["HasPool"] = (d["PoolArea"].fillna(0) > 0).astype(int)
    d["HasMiscVal"] = (d["MiscVal"].fillna(0) > 0).astype(int)
    d["HasAlley"] = d["Alley"].notna().astype(int)

    # Aggregations
    d["TotalBath_equiv"] = (
        d["FullBath"].fillna(0)
        + d["BsmtFullBath"].fillna(0)
        + 0.5 * (d["HalfBath"].fillna(0) + d["BsmtHalfBath"].fillna(0))
    ).clip(upper=4)

    d["TotalPorchSF"] = (
        d["WoodDeckSF"].fillna(0)
        + d["OpenPorchSF"].fillna(0)
        + d["3SsnPorch"].fillna(0)
        + d["ScreenPorch"].fillna(0)
    )

    d["BedroomShare"] = (
        d["BedroomAbvGr"]
        / d["TotRmsAbvGrd"].replace(0, np.nan)
    ).replace([np.inf, -np.inf], np.nan)

    # Ordinal grouping
    d["OverallCondGrp"] = pd.cut(
        d["OverallCond"],
        bins=[0, 3, 8, 10],
        labels=["Low", "Medium", "High"],
        include_lowest=True,
    )

    d["TotRmsAbvGrd"] = d["TotRmsAbvGrd"].clip(upper=11)
    d["Fireplaces"] = d["Fireplaces"].clip(upper=2)
    d["GarageCars"] = d["GarageCars"].clip(upper=3)

    # Neighborhood ordinal
    d["NeighborhoodTier"] = d["Neighborhood"].map(NEIGHBORHOOD_TIERS).fillna("Medium")

    # Outlier caps
    d["MasVnrArea"] = d["MasVnrArea"].clip(upper=1200)
    d["TotalBsmtSF"] = d["TotalBsmtSF"].clip(upper=5000)
    d["GrLivArea"] = d["GrLivArea"].clip(upper=4000)

    # Drop source columns after aggregations or flags
    d = d.drop(columns=[c for c in DROP_AFTER_ENGINEERING if c in d.columns], errors="ignore")
    return d

train_fe = engineer_features(train_raw)
test_fe = engineer_features(test_raw)

print("train_fe shape:", train_fe.shape)
print("test_fe shape :", test_fe.shape)


train_fe shape: (1460, 72)
test_fe shape : (1459, 71)


In [11]:
#Cell 6: Categorical analysis

cat_cols = train_fe.select_dtypes(include=["object", "category"]).columns.tolist()
cat_profile = pd.DataFrame({
    "feature": cat_cols,
    "cardinality": [train_fe[c].nunique(dropna=False) for c in cat_cols],
    "missing_rate": [train_fe[c].isna().mean() for c in cat_cols],
    "top_freq": [train_fe[c].value_counts(dropna=False, normalize=True).iloc[0] for c in cat_cols],
}).sort_values(["cardinality", "missing_rate"], ascending=False)

# Mutual information for categorical features vs log-target
mi_rows = []
y_log = np.log1p(train_fe["SalePrice"])
for c in cat_cols:
    x_codes = pd.factorize(train_fe[c].astype(str), sort=True)[0]
    mi = mutual_info_regression(x_codes.reshape(-1, 1), y_log.values, discrete_features=True, random_state=SEED)[0]
    mi_rows.append((c, mi))

cat_mi = pd.DataFrame(mi_rows, columns=["feature", "mutual_info"]).sort_values("mutual_info", ascending=False)

print("Top categorical features by MI:")
display(cat_mi.head(15))
print("Categorical profile (head):")
display(cat_profile.head(15))

Top categorical features by MI:


,feature,mutual_info
6,Neighborhood,0.531616
42,NeighborhoodTier,0.453338
16,ExterQual,0.327918
19,BsmtQual,0.325342
28,KitchenQual,0.324875
32,GarageFinish,0.261419
30,FireplaceQu,0.209302
31,GarageType,0.200588
18,Foundation,0.196644
25,HeatingQC,0.160911


Categorical profile (head):


,feature,cardinality,missing_rate,top_freq
6,Neighborhood,25,0.000000,0.154110
14,Exterior2nd,16,0.000000,0.345205
13,Exterior1st,15,0.000000,0.352740
7,Condition1,9,0.000000,0.863014
39,SaleType,9,0.000000,0.867808
8,Condition2,8,0.000000,0.989726
10,HouseStyle,8,0.000000,0.497260
12,RoofMatl,8,0.000000,0.982192
31,GarageType,7,0.055479,0.595890
23,BsmtFinType2,7,0.026027,0.860274


In [12]:
# Cell 7: Rare category consolidation

X_full = train_fe.drop(columns=["SalePrice"]).copy()
X_test_full = test_fe.copy()

cat_cols = X_full.select_dtypes(include=["object", "category"]).columns.tolist()

rare_map = {}
for c in cat_cols:
    freq = X_full[c].astype(str).value_counts(normalize=True, dropna=False)
    rare_levels = set(freq[freq < 0.01].index)
    rare_map[c] = rare_levels

    X_full[c] = X_full[c].astype(str).apply(lambda v: "Rare" if v in rare_levels else v)
    X_test_full[c] = X_test_full[c].astype(str).apply(lambda v: "Rare" if v in rare_levels else v)

print(f"Rare-grouped categorical columns: {len(cat_cols)}")

Rare-grouped categorical columns: 43


In [13]:
# Cell 8: Multicollinearity control: drop one of highly correlated pairs

num_cols_all = X_full.select_dtypes(include=["number"]).columns.tolist()

corr_abs = X_full[num_cols_all].corr().abs()
upper = corr_abs.where(np.triu(np.ones(corr_abs.shape), k=1).astype(bool))

corr_threshold = 0.90
to_drop_corr = [c for c in upper.columns if any(upper[c] > corr_threshold)]

# Protect core features we explicitly want to keep
protect = {"GrLivArea", "TotalBsmtSF", "OverallQual", "GarageCars", "TotalBath_equiv"}
to_drop_corr = [c for c in to_drop_corr if c not in protect]

X_model = X_full.drop(columns=to_drop_corr, errors="ignore").copy()
X_test_model = X_test_full.drop(columns=to_drop_corr, errors="ignore").copy()

print("Dropped due to multicollinearity:", to_drop_corr)
print("X_model shape:", X_model.shape)

Dropped due to multicollinearity: ['HouseAge']
X_model shape: (1460, 70)


In [16]:
# Cell 9: Preprocessing pipelines

num_cols = X_model.select_dtypes(include=["number"]).columns.tolist()
cat_cols = X_model.select_dtypes(exclude=["number"]).columns.tolist()

log_num_cols = [c for c in LOG_ROBUST_NUM_CANDIDATES if c in num_cols]
std_num_cols = [c for c in num_cols if c not in log_num_cols]

ordinal_cat_cols = [c for c in ["NeighborhoodTier", "OverallCondGrp"] if c in cat_cols]
nominal_cat_cols = [c for c in cat_cols if c not in ordinal_cat_cols]

ordinal_categories = []
for c in ordinal_cat_cols:
    if c == "NeighborhoodTier":
        ordinal_categories.append(["VeryLow", "Low", "Medium", "High", "VeryHigh"])
    elif c == "OverallCondGrp":
        ordinal_categories.append(["Low", "Medium", "High"])

log_num_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("log1p", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
    ("scaler", RobustScaler()),
])

std_num_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

if ordinal_cat_cols:
    ordinal_cat_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OrdinalEncoder(categories=ordinal_categories, handle_unknown="use_encoded_value", unknown_value=-1)),
    ])

nominal_cat_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    # drop='first' снижает dummy multicollinearity
    ("encoder", OneHotEncoder(handle_unknown="ignore", drop="first")),
])

transformers = []
if log_num_cols:
    transformers.append(("log_num", log_num_pipe, log_num_cols))
if std_num_cols:
    transformers.append(("std_num", std_num_pipe, std_num_cols))
if ordinal_cat_cols:
    transformers.append(("ord_cat", ordinal_cat_pipe, ordinal_cat_cols))
if nominal_cat_cols:
    transformers.append(("nom_cat", nominal_cat_pipe, nominal_cat_cols))

preprocessor = ColumnTransformer(transformers=transformers, remainder="drop")
print("log_num_cols:", len(log_num_cols))
print("std_num_cols:", len(std_num_cols))
print("ordinal_cat_cols:", ordinal_cat_cols)
print("nominal_cat_cols:", len(nominal_cat_cols))

log_num_cols: 7
std_num_cols: 20
ordinal_cat_cols: ['NeighborhoodTier', 'OverallCondGrp']
nominal_cat_cols: 41


In [47]:
from catboost import CatBoostRegressor

cv = KFold(n_splits=5, shuffle=True, random_state=SEED)

rmse_scorer = make_scorer(
    lambda yt, yp: np.sqrt(mean_squared_error(yt, yp)),
    greater_is_better=False,
)

# Build transformed matrices once and reuse below
X_train_matrix = preprocessor.fit_transform(X_model)
X_test_matrix = preprocessor.transform(X_test_model)

# CatBoost baseline on transformed matrix
baseline_model = CatBoostRegressor(
    random_seed=SEED,
    loss_function="RMSE",
    n_estimators=700,
    depth=6,
    learning_rate=0.05,
    verbose=0,
)

baseline_scores = -cross_val_score(
    baseline_model,
    X_train_matrix,
    y_log,
    cv=cv,
    scoring=rmse_scorer,
    n_jobs=1,
)

baseline_rmse_mean = baseline_scores.mean()
baseline_rmse_std = baseline_scores.std()

print(f"CatBoost baseline RMSE (log-target), CV mean: {baseline_rmse_mean:.5f}")
print(f"CatBoost baseline RMSE (log-target), CV std : {baseline_rmse_std:.5f}")

CatBoost baseline RMSE (log-target), CV mean: 0.12373
CatBoost baseline RMSE (log-target), CV std : 0.00853


In [23]:
!pip install FLAML

In [25]:
from flaml import AutoML

In [27]:
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 7.7 MB/s eta 0:00:00


In [28]:
# Cell 11: AutoML stage 1
X_train_matrix = preprocessor.fit_transform(X_model)
X_test_matrix = preprocessor.transform(X_test_model)

# Explicitly evaluate required models + a few extras
flaml_estimators = [
    "catboost",   # required
    "lgbm",       # required
    "rf",         # required
    "extra_tree",
    "xgboost",
    "xgb_limitdepth",
]
mandatory_estimators = {"catboost", "lgbm", "rf"}

leaderboard_rows = []
best_result = None

for est_name in flaml_estimators:
    automl = AutoML()
    settings = {
        "time_budget": 90,
        "task": "regression",
        "metric": "rmse",
        "eval_method": "cv",
        "n_splits": 3,
        "seed": SEED,
        "estimator_list": [est_name],
        "verbose": 0,
    }

    try:
        automl.fit(X_train=X_train_matrix, y_train=y_log, **settings)
        rmse = float(automl.best_loss)
        config = dict(automl.best_config) if automl.best_config else {}

        leaderboard_rows.append(
            {
                "estimator": est_name,
                "rmse": rmse,
                "status": "ok",
                "error": None,
                "best_config": config,
            }
        )

        print(f"FLAML | {est_name:14s} RMSE: {rmse:.5f}")

        if best_result is None or rmse < best_result["rmse"]:
            best_result = {
                "estimator": est_name,
                "rmse": rmse,
                "best_config": config,
            }

    except Exception as e:
        leaderboard_rows.append(
            {
                "estimator": est_name,
                "rmse": np.nan,
                "status": "failed",
                "error": str(e)[:200],
                "best_config": {},
            }
        )
        print(f"FLAML | {est_name:14s} FAILED: {str(e)[:120]}")

flaml_leaderboard = pd.DataFrame(leaderboard_rows).sort_values("rmse", na_position="last")
display(flaml_leaderboard[["estimator", "rmse", "status", "error"]])

successful_estimators = set(flaml_leaderboard.loc[flaml_leaderboard["status"] == "ok", "estimator"])
missing_mandatory = sorted(list(mandatory_estimators - successful_estimators))
assert len(missing_mandatory) == 0, f"Mandatory estimators not evaluated successfully: {missing_mandatory}"

best_stage1_name = best_result["estimator"]
best_stage1_rmse = best_result["rmse"]
best_stage1_params = best_result["best_config"]

print("\nStage-1 best estimator:", best_stage1_name)
print("Stage-1 best RMSE:", round(best_stage1_rmse, 5))
print("Stage-1 best config:")
pprint(best_stage1_params)

FLAML | catboost       RMSE: 0.12694
FLAML | lgbm           RMSE: 0.12634
FLAML | rf             RMSE: 0.13828
FLAML | extra_tree     RMSE: 0.13917
FLAML | xgboost        RMSE: 0.12403
FLAML | xgb_limitdepth RMSE: 0.12049


,estimator,rmse,status,error
5,xgb_limitdepth,0.120487,ok,None
4,xgboost,0.124026,ok,None
1,lgbm,0.126345,ok,None
0,catboost,0.126937,ok,None
2,rf,0.138279,ok,None
3,extra_tree,0.139167,ok,None



Stage-1 best estimator: xgb_limitdepth
Stage-1 best RMSE: 0.12049
Stage-1 best config:
{'colsample_bylevel': np.float64(0.4229333136636264),
 'colsample_bytree': np.float64(0.801112830671241),
 'learning_rate': np.float64(0.03553493436124587),
 'max_depth': 2,
 'min_child_weight': np.float64(0.026263196124310568),
 'n_estimators': 653,
 'reg_alpha': 0.0009765625,
 'reg_lambda': np.float64(0.28384509698948235),
 'subsample': np.float64(0.9005885316832697)}


In [37]:
# Cell 12 (2): Neural network benchmark
X_train_dense = X_train_matrix.toarray() if hasattr(X_train_matrix, "toarray") else np.asarray(X_train_matrix)
X_test_dense = X_test_matrix.toarray() if hasattr(X_test_matrix, "toarray") else np.asarray(X_test_matrix)

nn_base = MLPRegressor(
    random_state=SEED,
    max_iter=3000,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=60,
)

nn_param_dist = {
    "hidden_layer_sizes": [(128,), (256,), (128, 64), (256, 128), (256, 128, 64), (128, 64, 32)],
    "activation": ["relu", "tanh", ""],
    "alpha": [1e-5, 1e-4, 1e-3, 1e-2],
    "learning_rate_init": [5e-4, 1e-3, 3e-3, 1e-2],
    "batch_size": [32, 64, 128, 256],
}

nn_search = RandomizedSearchCV(
    estimator=nn_base,
    param_distributions=nn_param_dist,
    n_iter=12,
    scoring=rmse_scorer,
    cv=3,
    random_state=SEED,
    n_jobs=1,
    verbose=1,
)
nn_search.fit(X_train_dense, y_log)

nn_rmse = -nn_search.best_score_
nn_best_params = nn_search.best_params_
nn_best_model = nn_search.best_estimator_

print("Neural Net best RMSE:", round(nn_rmse, 5))
print("Neural Net best params:")
pprint(nn_best_params)

Fitting 3 folds for each of 12 candidates, totalling 36 fits
Neural Net best RMSE: 0.1358
Neural Net best params:
{'activation': 'tanh',
 'alpha': 0.01,
 'batch_size': 32,
 'hidden_layer_sizes': (128, 64),
 'learning_rate_init': 0.003}


In [40]:
from xgboost import XGBRegressor
best_stage1_params = {
    "colsample_bylevel": 0.4229333136636264,
    "colsample_bytree": 0.801112830671241,
    "learning_rate": 0.03553493436124587,
    "max_depth": 2,
    "min_child_weight": 0.026263196124310568,
    "n_estimators": 653,
    "reg_alpha": 0.0009765625,
    "reg_lambda": 0.28384509698948235,
    "subsample": 0.9005885316832697,
}

stage1_model = XGBRegressor(
    objective="reg:squarederror",
    random_state=SEED,
    n_jobs=1,
    tree_method="hist",
    n_estimators=best_stage1_params["n_estimators"],
    max_depth=best_stage1_params["max_depth"],
    learning_rate=best_stage1_params["learning_rate"],
    colsample_bytree=best_stage1_params["colsample_bytree"],
    colsample_bylevel=best_stage1_params["colsample_bylevel"],
    subsample=best_stage1_params["subsample"],
    min_child_weight=best_stage1_params["min_child_weight"],
    reg_alpha=best_stage1_params["reg_alpha"],
    reg_lambda=best_stage1_params["reg_lambda"],
)

stage1_scores = -cross_val_score(
    stage1_model,
    X_train_matrix,
    y_log,
    cv=5,
    scoring=rmse_scorer,
    n_jobs=1,
)
stage1_cv_rmse = float(stage1_scores.mean())

print(f"Stage-1 fixed model CV RMSE: {stage1_cv_rmse:.5f}")

Stage-1 fixed model CV RMSE: 0.12074


In [42]:

base_est = XGBRegressor(
    objective="reg:squarederror",
    random_state=SEED,
    n_jobs=1,
    tree_method="hist",
)

finetune_params = {
    "n_estimators": [450, 653, 850, 1100, 1400],
    "max_depth": [2, 3, 4, 5],
    "learning_rate": [0.01, 0.02, 0.03553493436124587, 0.05, 0.08],
    "colsample_bytree": [0.6, 0.8, 1.0],
    "colsample_bylevel": [0.3, 0.4229333136636264, 0.6, 0.8, 1.0],
    "subsample": [0.7, 0.85, 0.9005885316832697, 1.0],
    "min_child_weight": [0.01, 0.026263196124310568, 0.1, 0.3, 1.0],
    "reg_alpha": [0.0, 0.0009765625, 0.01, 0.1, 0.5],
    "reg_lambda": [0.1, 0.28384509698948235, 0.5, 1.0, 2.0],
}

finetuner = RandomizedSearchCV(
    estimator=base_est,
    param_distributions=finetune_params,
    n_iter=20,
    scoring=rmse_scorer,
    cv=5,
    random_state=SEED,
    n_jobs=1,
    verbose=1,
)
finetuner.fit(X_train_matrix, y_log)

final_model = finetuner.best_estimator_
fine_tuned_rmse = -finetuner.best_score_
fine_tuned_params = finetuner.best_params_

print("Fine-tuned RMSE:", round(fine_tuned_rmse, 5))
print("Fine-tuned params:")
pprint(fine_tuned_params)

Fitting 5 folds for each of 20 candidates, totalling 100 fits
Fine-tuned RMSE: 0.11699
Fine-tuned params:
{'colsample_bylevel': 0.3,
 'colsample_bytree': 0.8,
 'learning_rate': 0.02,
 'max_depth': 3,
 'min_child_weight': 1.0,
 'n_estimators': 1100,
 'reg_alpha': 0.0009765625,
 'reg_lambda': 0.1,
 'subsample': 0.7}


In [44]:
# Cell 13: Train final model on full train and predict test

final_model.fit(X_train_matrix, y_log)
pred_log = final_model.predict(X_test_matrix)
pred_price = np.expm1(pred_log)

submission = pd.read_csv("/content/sample_submission.csv")
submission["SalePrice"] = pred_price

submission.to_csv("/content/final_submission.csv", index=False)

submission.head(10)


,Id,SalePrice
0,1461,121400.054688
1,1462,152560.750000
2,1463,175958.578125
3,1464,191100.625000
4,1465,181741.453125
5,1466,171212.218750
6,1467,177976.890625
7,1468,159624.703125
8,1469,183147.312500
9,1470,120836.601562


In [46]:
# Cell additional (report):

report_lines = []
report_lines.append("=== DATA REPORT ===")
report_lines.append(f"Train rows: {len(train_raw)}, Test rows: {len(test_raw)}")
report_lines.append(f"Engineered train columns: {train_fe.shape[1]}")
report_lines.append(f"Categorical columns analyzed: {len(cat_cols)}")
report_lines.append(f"Dropped high-correlation numeric columns (>0.90): {len(to_drop_corr)}")
report_lines.append(f"Baseline RMSE (log-target): {baseline_rmse_mean:.5f} +/- {baseline_rmse_std:.5f}")
report_lines.append(f"AutoML checked estimators: {', '.join(flaml_estimators)}")
report_lines.append(f"Stage-1 best model: {best_stage1_name} | RMSE: {best_stage1_rmse:.5f}")
report_lines.append(f"Fine-tuned RMSE: {fine_tuned_rmse:.5f}")
report_lines.append("")
report_lines.append("Multicollinearity controls:")
report_lines.append("1) OneHotEncoder(drop='first') for nominal categorical features")
report_lines.append("2) Correlation-based feature pruning for numeric features")
report_lines.append("3) Ordinal features encoded by explicit order")

print("\n".join(report_lines))
display(flaml_leaderboard)

=== DATA REPORT ===
Train rows: 1460, Test rows: 1459
Engineered train columns: 72
Categorical columns analyzed: 43
Dropped high-correlation numeric columns (>0.90): 1
Baseline RMSE (log-target): 0.12734 +/- 0.01120
AutoML checked estimators: catboost, lgbm, rf, extra_tree, xgboost, xgb_limitdepth
Stage-1 best model: xgb_limitdepth | RMSE: 0.12049
Fine-tuned RMSE: 0.11699

Multicollinearity controls:
1) OneHotEncoder(drop='first') for nominal categorical features
2) Correlation-based feature pruning for numeric features
3) Ordinal features encoded by explicit order


,estimator,rmse,status,error,best_config
5,xgb_limitdepth,0.120487,ok,None,"{'n_estimators': 653, 'max_depth': 2, 'min_chi..."
4,xgboost,0.124026,ok,None,"{'n_estimators': 513, 'max_leaves': 9, 'min_ch..."
1,lgbm,0.126345,ok,None,"{'n_estimators': 242, 'num_leaves': 7, 'min_ch..."
0,catboost,0.126937,ok,None,"{'early_stopping_rounds': 11, 'learning_rate':..."
2,rf,0.138279,ok,None,"{'n_estimators': 248, 'max_features': 0.430809..."
3,extra_tree,0.139167,ok,None,"{'n_estimators': 145, 'max_features': 0.840735..."
